# Lab 2d · Do notebook à API
**Oficina do Builder · Winovoito** — Bancada 2 · ML aplicado

Um modelo que só existe no notebook não ajuda ninguém. Aqui o modelo de churn do Lab 2b vira uma API:

- `POST /prever_churn` → probabilidade, decisão pelo limiar, os 3 fatores que mais pesaram e a versão do modelo;
- `GET /saude` → versão e data do modelo.

> **Segurança:** arquivos de modelo (joblib/pickle) podem executar código ao serem abertos. Só carregue modelos que você mesmo gerou ou de fonte verificada. A função `churn.carregar` confere o hash antes de abrir.

In [ ]:
# Setup: baixa o material aberto da Oficina (rode uma vez)
REPO_URL = "https://github.com/winovoito/oficina-do-builder-aberto"  # endereço definitivo a confirmar
import os, sys
if not os.path.exists("oficina-do-builder-aberto"):
    !git clone -q {REPO_URL}
sys.path.insert(0, "oficina-do-builder-aberto")
%pip install -q -r oficina-do-builder-aberto/requirements.txt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)

## 1 · O modelo salvo no Lab 2b
Se você não tem a pasta `modelos/churn` (por exemplo, abriu este notebook direto), a célula gera o modelo de referência.

In [ ]:
from pathlib import Path
from oficina import churn
from oficina.sinteticos import gerar_fluxo

PASTA = Path("modelos/churn")
if not (PASTA / "metadados.json").exists():
    churn.rodar_referencia(gerar_fluxo(), PASTA)
modelo, meta = churn.carregar(PASTA)
meta

## 2 · O formato de entrada, com Pydantic
Tipos, faixas válidas e os planos permitidos. Campo a mais também é recusado.

In [ ]:
from oficina.api_churn import Cliente
import inspect
print(inspect.getsource(Cliente))
Cliente.model_json_schema()["properties"]["tempo_casa_meses"]

## 3 · A API
Leia `oficina/api_churn.py`: são poucas linhas. A função `criar_app` carrega o modelo, confere o hash e declara os dois endereços.

In [ ]:
from fastapi.testclient import TestClient
from oficina.api_churn import criar_app

treino, _ = churn.dividir(gerar_fluxo())
app = criar_app(PASTA, referencia=treino)
api = TestClient(app)
api.get("/saude").json()

## 4 · Testes: válido, inválido e no limite
Uma API que só funciona no caminho feliz quebra no primeiro dia.

In [ ]:
valido = {"plano": "basico", "mensalidade": 49, "tempo_casa_meses": 4, "chamados_90d": 5,
          "uso_semanal_horas": 2, "dias_atraso_pagamento": 30, "nps": 4}

r = api.post("/prever_churn", json=valido)
assert r.status_code == 200 and r.json()["versao_modelo"] == meta["versao"]
print("✅ válido:", r.json())

r = api.post("/prever_churn", json={**valido, "tempo_casa_meses": -3})
assert r.status_code == 422
print("✅ inválido devolve 422:", r.json()["detail"][0]["msg"])

r = api.post("/prever_churn", json={**valido, "tempo_casa_meses": 0, "chamados_90d": 0,
                                    "dias_atraso_pagamento": 0, "nps": None})
assert r.status_code == 200
print("✅ no limite:", r.json()["probabilidade"])

### Exercício · Mais um teste
Escreva um teste para um cliente com plano que não existe (`"ouro"`) e outro com um campo a mais (`"cpf"`). Os dois devem devolver 422. Por que recusar um campo a mais é uma boa ideia? (Pense em dados pessoais que ninguém pediu.)

In [ ]:
# TODO
assert api.post("/prever_churn", json={**valido, "plano": "ouro"}).status_code == 422
assert api.post("/prever_churn", json={**valido, "cpf": "000.000.000-00"}).status_code == 422
print("✅ seus testes passaram")

## 5 · Rodar de verdade
**No Colab**, para testar dentro do notebook, a API sobe em segundo plano:

In [ ]:
import threading, time, uvicorn, requests
servidor = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=servidor.run, daemon=True).start()
time.sleep(2)
print(requests.post("http://127.0.0.1:8000/prever_churn", json=valido).json())
servidor.should_exit = True

**No seu computador**, depois de rodar o Lab 2b:

```bash
pip install -r requirements.txt
OFICINA_MODELO_PASTA=modelos/churn uvicorn oficina.api_churn:app --reload
# abra http://127.0.0.1:8000/docs para testar pelo navegador
```

Colocar a API num servidor na nuvem é o próximo passo natural, mas depende da infraestrutura que você escolher e fica fora do escopo da Oficina.

**Próximo:** Bancada 3 · Prompts como código. Na Bancada 5, esta API vira a ferramenta `prever_churn` do agente.